# Importing Libraries

In [9]:
import pandas as pd
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime, timezone

# Setup Configurations

In [5]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
COMPANIES_PATH = DATA_DIR / "companies.csv"

print(f"Base directory: {BASE_DIR} | Base directory exists: {BASE_DIR.exists()}")
print(f"Companies path: {COMPANIES_PATH} |File exists: {COMPANIES_PATH.exists()}")

Base directory: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai | Base directory exists: True
Companies path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\companies.csv |File exists: True


## Load the registry

In [7]:
companies_df = pd.read_csv(COMPANIES_PATH)
companies_df.head()

,company,career_url,ats_type,ats_slug,workday_tenant,workday_site,workday_server,keywords,locations,priority
0,OpenAI,https://openai.com/careers/search/,ashby,openai,NaN,NaN,NaN,machine learning;data scientist;ai engineer;so...,United States;Remote,high
1,Anthropic,https://www.anthropic.com/jobs,greenhouse,anthropic,NaN,NaN,NaN,machine learning;data scientist;ai engineer;so...,United States;Remote,high
2,Netflix,https://jobs.netflix.com/,lever,netflix,NaN,NaN,NaN,machine learning;data scientist;software engineer,United States;Remote,medium
3,Workday,https://workday.wd5.myworkdayjobs.com/Workday,workday,NaN,workday,Workday,wd5,machine learning;data scientist;software engineer,United States;Remote,medium


# Helper Functions

In [8]:
Headers ={
    "User-Agent": "Mozilla/5.0 JobScout/0.1"
}

In [10]:
def clean_html(html_content):
    if not html_content:
        return ""
    
    soup = BeautifulSoup(str(html_content), "html.parser")
    return " ".join(soup.get_text(" ").split())

## normalize tags

In [12]:
def first_available(data, keys, default=""):
    """
    Return the first non-empty value from a dictionary.
    """
    for key in keys:
        value = data.get(key)
        if value not in [None, "", [], {}]:
            return value
    return default



In [13]:

def normalize_location(value):
    """
    Convert different location formats into a readable string.
    """
    if not value:
        return ""

    if isinstance(value, str):
        return value

    if isinstance(value, dict):
        return first_available(value, ["name", "location", "city", "text"])

    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(first_available(item, ["name", "location", "city", "text"]))
        return ", ".join([p for p in parts if p])

    return str(value)

## Greenhouse

In [15]:
def fetch_greenhouse(company, ats_slug):
    url = f"https://boards-api.greenhouse.io/v1/boards/{ats_slug}/jobs?content=true"

    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data.get("jobs", []):
        location = normalize_location(
            first_available(job, [
                "location",
                "offices"
            ])
        )

        posted_date = first_available(job, [
            "first_published",
            "published_at",
            "updated_at",
            "created_at"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["title"]),
            "location": location,
            "job_url": first_available(job, ["absolute_url"]),
            "description": clean_html(first_available(job, ["content"])),
            "ats_type": "greenhouse",
            "external_job_id": str(first_available(job, ["id"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Ashby fetcher

In [14]:
def fetch_ashby(company, ats_slug):
    url = f"https://api.ashbyhq.com/posting-api/job-board/{ats_slug}?includeCompensation=true"

    response = requests.get(url, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data.get("jobs", []):
        location = normalize_location(
            first_available(job, [
                "location",
                "locations",
                "locationName",
                "address",
                "office",
                "offices"
            ])
        )

        description = first_available(job, [
            "descriptionHtml",
            "descriptionPlain",
            "description",
            "jobDescription"
        ])

        posted_date = first_available(job, [
            "publishedAt",
            "publishedDate",
            "postedDate",
            "createdAt",
            "updatedAt"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["title", "name"]),
            "location": location,
            "job_url": first_available(job, ["jobUrl", "applyUrl", "url"]),
            "description": clean_html(description),
            "ats_type": "ashby",
            "external_job_id": str(first_available(job, ["id", "jobId"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Lever

In [16]:
def extract_lever_description(job):
    description_parts = []

    # Main description
    description_parts.append(first_available(job, [
        "descriptionPlain",
        "description",
    ]))

    # Additional section
    description_parts.append(first_available(job, [
        "additionalPlain",
        "additional",
    ]))

    # Important: Lever stores responsibilities/requirements here
    for section in job.get("lists", []):
        section_title = section.get("text", "")
        section_content = clean_html(section.get("content", ""))

        if section_title or section_content:
            description_parts.append(f"{section_title}: {section_content}")

    return " ".join([part for part in description_parts if part])

In [ ]:
def fetch_lever(company, ats_slug):
    url = f"https://api.lever.co/v0/postings/{ats_slug}?mode=json"

    response = requests.get(url, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data:
        categories = job.get("categories", {}) or {}

        location_parts = [
            first_available(categories, ["location", "team", "department"]),
            first_available(job, ["country"]),
            first_available(job, ["workplaceType"]),
        ]
        location = " ".join(str(part) for part in location_parts if part)

        description = extract_lever_description(job)

        posted_date = first_available(job, [
            "createdAt",
            "updatedAt",
            "created_at",
            "updated_at"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["text", "title"]),
            "location": location,
            "job_url": first_available(job, ["hostedUrl", "applyUrl", "url"]),
            "description": description,
            "ats_type": "lever",
            "external_job_id": str(first_available(job, ["id"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs